In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "true"

DATA_DIR = Path("/kaggle/input/competitions/ai-generated-text-detection-hw3-ucu-ai-2026")
BASE_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/01-eda-split-baseline")
MODEL_NAME = "microsoft/mdeberta-v3-base"
CLASSES = ["human", "fusion", "ai"]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}

# 1. Перевірка GPU та версій бібліотек
print(f"PyTorch: {torch.__version__} | Transformers: {transformers.__version__}")
n_gpu = torch.cuda.device_count()
assert n_gpu >= 1, "GPU not detected! Check Accelerator settings."
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB")

for pkg in ("sentencepiece", "accelerate"):
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'installed')}")
    except ImportError:
        print(f"WARNING: {pkg} is NOT installed!")

# 2. Перевірка шляхів і файлів із Ноутбука 1
assert DATA_DIR.exists(), f"Missing DATA_DIR: {DATA_DIR}"
assert BASE_DIR.exists(), f"Missing BASE_DIR: {BASE_DIR}"
print("\nFiles in BASE_DIR:", sorted(p.name for p in BASE_DIR.iterdir()))

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
splits_df = pd.read_parquet(BASE_DIR / "folds.parquet")
base_oof = pd.read_parquet(BASE_DIR / "baseline_oof_fold0.parquet")

# Жорстка звірка ID і порядку класів із бейзлайном
assert (train["id"].to_numpy() == splits_df["id"].to_numpy()).all(), "train vs folds ID mismatch!"
train["fold"] = splits_df["fold"].to_numpy()
train["y"] = train["label"].map(LABEL2ID).astype(np.int64)

va_df = train[train["fold"] == 0].reset_index(drop=True)
assert (va_df["id"].to_numpy() == base_oof["id"].to_numpy()).all(), "Val fold 0 vs baseline OOF ID mismatch!"
assert (va_df["y"].to_numpy() == base_oof["y_true"].to_numpy()).all(), "Label mapping mismatch with baseline!"
print(f"\nData OK | Train (folds 1-4): {(train['fold'] != 0).sum()} | Val (fold 0): {len(va_df)} | Test: {len(test)}")

# 3. Завантаження токенізатора та аналіз довжин на підвибірці (5000 текстів)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"\nTokenizer loaded: {tokenizer.__class__.__name__} | vocab_size={tokenizer.vocab_size}")

sample = train.sample(n=5000, random_state=42).reset_index(drop=True)
enc = tokenizer(
    sample["text"].tolist(),
    add_special_tokens=False,
    truncation=False,
    return_length=True,
)
sample["n_tokens"] = enc["length"]

pcts = np.percentile(sample["n_tokens"], [50, 75, 90, 95, 99])
print(
    f"\nToken length percentiles (n=5000): "
    f"p50={pcts[0]:.0f} | p75={pcts[1]:.0f} | p90={pcts[2]:.0f} | p95={pcts[3]:.0f} | p99={pcts[4]:.0f}"
)
for lim in (256, 384, 512, 768):
    print(f"  > {lim} tokens: {(sample['n_tokens'] > lim).mean() * 100:.1f}%")

print("\n--- Median & % > 512 tokens by Label ---")
print(
    sample.groupby("label")["n_tokens"]
    .agg(median="median", p90=lambda s: np.percentile(s, 90), pct_gt_512=lambda s: (s > 512).mean() * 100)
    .round(1)
)

print("\n--- Median & % > 512 tokens by Domain ---")
print(
    sample.groupby("domain")["n_tokens"]
    .agg(median="median", p90=lambda s: np.percentile(s, 90), pct_gt_512=lambda s: (s > 512).mean() * 100)
    .round(1)
)

PyTorch: 2.11.0+cu130 | Transformers: 5.20.2
GPU 0: Tesla T4 | VRAM: 14.6 GB
GPU 1: Tesla T4 | VRAM: 14.6 GB
sentencepiece: 0.2.2
accelerate: 1.15.0

Files in BASE_DIR: ['.virtual_documents', '__notebook__.ipynb', '__output__.json', '__results__.html', '__results___files', 'baseline_experiments.csv', 'baseline_oof_fold0.parquet', 'baseline_test_preds.parquet', 'custom.css', 'folds.parquet', 'submission_baseline.csv']

Data OK | Train (folds 1-4): 217762 | Val (fold 0): 54590 | Test: 25000


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spm.model: downloading bytes:           |  0.00B            


Tokenizer loaded: DebertaV2Tokenizer | vocab_size=250101

Token length percentiles (n=5000): p50=764 | p75=1143 | p90=1594 | p95=2049 | p99=3742
  > 256 tokens: 87.2%
  > 384 tokens: 78.1%
  > 512 tokens: 69.3%
  > 768 tokens: 49.7%

--- Median & % > 512 tokens by Label ---
        median     p90  pct_gt_512
label                             
ai       789.0  1456.0        73.3
fusion   718.0  1842.8        66.4
human    743.5  1720.0        65.6

--- Median & % > 512 tokens by Domain ---
                        median     p90  pct_gt_512
domain                                            
arxiv_papers            1491.0  3733.2        94.0
blog_authorship_corpus   217.0   681.2        18.5
bundestag                947.5  1493.5        86.8
cnn_news                 879.0  1606.9        83.0
euro_court_cases        1040.0  2735.7        84.8
gutenberg               1222.0  1936.2        95.1
house_of_commons         864.0  1652.8        75.4
student_essays           560.5   926.2        5

In [2]:
%%writefile /kaggle/working/train_mdeberta.py
import argparse
import gc
import math
import os
import time
from contextlib import nullcontext
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)

DATA_DIR = Path("/kaggle/input/competitions/ai-generated-text-detection-hw3-ucu-ai-2026")
BASE_DIR = Path("/kaggle/input/notebooks/oleksandrtsybrivskyi/01-eda-split-baseline")
OUT_DIR = Path("/kaggle/working")

MODEL_NAME = "microsoft/mdeberta-v3-base"
CLASSES = ["human", "fusion", "ai"]
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}
ID2LABEL = {i: c for i, c in enumerate(CLASSES)}


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class HeadTailDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len: int = 512, labels=None):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.labels = labels
        self.cls_id = tokenizer.cls_token_id
        self.sep_id = tokenizer.sep_token_id
        assert self.cls_id is not None and self.sep_id is not None

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        ids = self.tokenizer.encode(self.texts[idx], add_special_tokens=False)
        budget = self.max_len - 2
        if len(ids) > budget:
            head = budget // 2
            tail = budget - head
            ids = ids[:head] + ids[-tail:]
        input_ids = [self.cls_id] + ids + [self.sep_id]
        item = {"idx": idx, "input_ids": input_ids}
        if self.labels is not None:
            item["label"] = int(self.labels[idx])
        return item


def make_collate_fn(pad_id: int, max_len: int):
    def collate_fn(batch):
        max_l = max(len(x["input_ids"]) for x in batch)
        # Кратність 8 прискорює Tensor Cores на T4
        max_l = min(max_len, ((max_l + 7) // 8) * 8)

        bs = len(batch)
        input_ids = torch.full((bs, max_l), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((bs, max_l), dtype=torch.long)
        idxs = torch.empty(bs, dtype=torch.long)

        for i, item in enumerate(batch):
            seq = item["input_ids"]
            L = len(seq)
            input_ids[i, :L] = torch.tensor(seq, dtype=torch.long)
            attention_mask[i, :L] = 1
            idxs[i] = item["idx"]

        out = {"idx": idxs, "input_ids": input_ids, "attention_mask": attention_mask}
        if "label" in batch[0]:
            out["labels"] = torch.tensor([x["label"] for x in batch], dtype=torch.long)
        return out

    return collate_fn


@torch.no_grad()
def predict_ddp(model, loader, n_total: int, device, rank: int, world_size: int):
    model.eval()
    local_idxs, local_logits = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        local_idxs.append(batch["idx"].numpy())
        local_logits.append(logits.float().cpu().numpy())

    local_idxs = np.concatenate(local_idxs, axis=0)
    local_logits = np.concatenate(local_logits, axis=0)

    gathered = [None] * world_size
    dist.all_gather_object(gathered, (local_idxs, local_logits))

    if rank != 0:
        return None

    full_logits = np.zeros((n_total, len(CLASSES)), dtype=np.float32)
    seen = np.zeros(n_total, dtype=bool)
    for g_idx, g_log in gathered:
        full_logits[g_idx] = g_log
        seen[g_idx] = True

    assert seen.all(), f"Missing predictions for {(~seen).sum()} rows!"
    assert np.isfinite(full_logits).all(), "Non-finite logits detected!"
    return full_logits


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--smoke_test", action="store_true")
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--eval_batch_size", type=int, default=32)
    parser.add_argument("--grad_accum", type=int, default=2)
    parser.add_argument("--max_len", type=int, default=512)
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--warmup_ratio", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--evals_per_epoch", type=int, default=2)
    args = parser.parse_args()

    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)
    set_seed(args.seed + rank)

    # 1. Завантаження даних та спліту
    train_df = pd.read_csv(DATA_DIR / "train.csv")
    test_df = pd.read_csv(DATA_DIR / "test.csv")
    splits_df = pd.read_parquet(BASE_DIR / "folds.parquet")
    assert (train_df["id"].to_numpy() == splits_df["id"].to_numpy()).all()

    train_df["fold"] = splits_df["fold"].to_numpy()
    train_df["y"] = train_df["label"].map(LABEL2ID).astype(np.int64)

    tr_df = train_df[train_df["fold"] != 0].reset_index(drop=True)
    va_df = train_df[train_df["fold"] == 0].reset_index(drop=True)
    full_train_len = len(tr_df)

    if args.smoke_test:
        tr_df = tr_df.sample(n=2000, random_state=args.seed).reset_index(drop=True)
        va_df = va_df.sample(n=500, random_state=args.seed).reset_index(drop=True)
        test_df = test_df.head(500).reset_index(drop=True)

    if rank == 0:
        mode = "SMOKE TEST" if args.smoke_test else "FULL RUN"
        eff_bs = args.batch_size * args.grad_accum * world_size
        print(
            f"[{mode}] GPUs={world_size} | max_len={args.max_len} (head+tail) | "
            f"per_gpu_bs={args.batch_size} | grad_accum={args.grad_accum} | eff_bs={eff_bs} | lr={args.lr}"
        )
        print(f"Rows -> Train: {len(tr_df)} | Val: {len(va_df)} | Test: {len(test_df)}")

    # 2. Токенізатор і DataLoader-и
    if rank != 0:
        dist.barrier()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if rank == 0:
        dist.barrier()
    tokenizer.model_max_length = 1_000_000
    collate_fn = make_collate_fn(tokenizer.pad_token_id, args.max_len)

    tr_ds = HeadTailDataset(tr_df["text"].tolist(), tokenizer, args.max_len, tr_df["y"].to_numpy())
    va_ds = HeadTailDataset(va_df["text"].tolist(), tokenizer, args.max_len, va_df["y"].to_numpy())
    te_ds = HeadTailDataset(test_df["text"].tolist(), tokenizer, args.max_len, None)

    tr_sampler = DistributedSampler(tr_ds, num_replicas=world_size, rank=rank, shuffle=True, seed=args.seed)
    va_sampler = DistributedSampler(va_ds, num_replicas=world_size, rank=rank, shuffle=False)
    te_sampler = DistributedSampler(te_ds, num_replicas=world_size, rank=rank, shuffle=False)

    tr_loader = DataLoader(
        tr_ds, batch_size=args.batch_size, sampler=tr_sampler,
        num_workers=2, pin_memory=True, collate_fn=collate_fn, drop_last=False
    )
    va_loader = DataLoader(
        va_ds, batch_size=args.eval_batch_size, sampler=va_sampler,
        num_workers=2, pin_memory=True, collate_fn=collate_fn, drop_last=False
    )
    te_loader = DataLoader(
        te_ds, batch_size=args.eval_batch_size, sampler=te_sampler,
        num_workers=2, pin_memory=True, collate_fn=collate_fn, drop_last=False
    )

    # 3. Модель, оптимізатор, шедулер
    if rank != 0:
        dist.barrier()
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(CLASSES))
    if rank == 0:
        dist.barrier()
    model = model.float()
    assert all(p.dtype == torch.float32 for p in model.parameters()), "Master weights must be FP32!"
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.to(device)
    model = DDP(model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False)

    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias")
    grouped_params = [
        {
            "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": args.weight_decay,
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]
    optimizer = torch.optim.AdamW(grouped_params, lr=args.lr, eps=1e-6)

    steps_per_epoch = math.ceil(len(tr_loader) / args.grad_accum)
    total_opt_steps = max(1, int(math.ceil(steps_per_epoch * args.epochs)))
    warmup_steps = int(total_opt_steps * args.warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_opt_steps)

    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", init_scale=4096.0)

    eval_interval = max(1, steps_per_epoch // args.evals_per_epoch)
    best_f1 = -1.0
    best_oof_logits = None
    best_ckpt_path = OUT_DIR / ("mdeberta_smoke_best.pt" if args.smoke_test else "mdeberta_best.pt")

    # 4. Цикл тренування
    global_opt_step = 0
    nan_skips = 0
    t_train_start = time.time()
    n_epochs_int = math.ceil(args.epochs)

    for epoch in range(n_epochs_int):
        tr_sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        log_steps = 0
        t_log = time.time()

        for step, batch in enumerate(tr_loader, start=1):
            is_sync = (step % args.grad_accum == 0) or (step == len(tr_loader))
            ctx = nullcontext() if is_sync else model.no_sync()

            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            with ctx:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                    loss = criterion(logits.float(), labels) / args.grad_accum
                scaler.scale(loss).backward()

            running_loss += loss.item() * args.grad_accum
            log_steps += 1

            if is_sync:
                scale_before = scaler.get_scale()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                if scaler.get_scale() < scale_before:
                    nan_skips += 1
                else:
                    scheduler.step()

                global_opt_step += 1

                if rank == 0 and (global_opt_step % max(1, min(100, eval_interval)) == 0 or global_opt_step == 10):
                    dt = time.time() - t_log
                    lr_now = scheduler.get_last_lr()[0]
                    vram = torch.cuda.max_memory_allocated(device) / 1024**3
                    print(
                        f"Ep {epoch + 1} | step {global_opt_step}/{total_opt_steps} | "
                        f"loss={running_loss / log_steps:.4f} | lr={lr_now:.2e} | "
                        f"vram_peak={vram:.2f}GB | skips={nan_skips} | {dt:.1f}s",
                        flush=True,
                    )
                    running_loss, log_steps, t_log = 0.0, 0, time.time()

                do_eval = (global_opt_step % eval_interval == 0) or (global_opt_step >= total_opt_steps)
                if do_eval:
                    t_ev = time.time()
                    val_logits = predict_ddp(model, va_loader, len(va_ds), device, rank, world_size)
                    if rank == 0:
                        val_preds = val_logits.argmax(axis=1)
                        va_y = va_df["y"].to_numpy()
                        macro_f1 = f1_score(va_y, val_preds, average="macro")
                        per_cls = f1_score(va_y, val_preds, average=None, labels=[0, 1, 2])
                        print(
                            f">>> [EVAL step {global_opt_step}/{total_opt_steps}] "
                            f"Macro-F1={macro_f1:.4f} (H={per_cls[0]:.4f}, F={per_cls[1]:.4f}, A={per_cls[2]:.4f}) "
                            f"| eval_time={time.time() - t_ev:.1f}s",
                            flush=True,
                        )
                        if macro_f1 > best_f1:
                            best_f1 = macro_f1
                            best_oof_logits = val_logits.copy()
                            torch.save(model.module.state_dict(), best_ckpt_path)
                    dist.barrier()
                    model.train()

                if global_opt_step >= total_opt_steps:
                    break

    train_elapsed = time.time() - t_train_start

    # 5. Завантаження найкращого чекпоінта та інференс на test
    dist.barrier()
    state = torch.load(best_ckpt_path, map_location=device, weights_only=True)
    model.module.load_state_dict(state)
    t_te = time.time()
    te_logits = predict_ddp(model, te_loader, len(te_ds), device, rank, world_size)
    te_elapsed = time.time() - t_te

    if rank == 0:
        vram_peak = torch.cuda.max_memory_allocated(device) / 1024**3
        print(f"\n=== FINISHED | Best Val Macro-F1={best_f1:.4f} | Peak VRAM={vram_peak:.2f}GB | NaN skips={nan_skips} ===")

        if args.smoke_test:
            # Оцінка часу повної епохи на 217,762 рядках
            scale_tr = full_train_len / len(tr_df)
            est_ep_min = (train_elapsed * scale_tr) / 60.0
            est_te_min = (te_elapsed * (25000 / len(test_df))) / 60.0
            print(
                f"[ETA PROJECTION] 1 full epoch train (~217.8k rows) ≈ {est_ep_min:.1f} min "
                f"| full test inference (25k rows) ≈ {est_te_min:.1f} min"
            )
            if best_ckpt_path.exists():
                best_ckpt_path.unlink()
        else:
            va_y = va_df["y"].to_numpy()
            va_preds = best_oof_logits.argmax(axis=1)
            print("\n--- Classification Report (Val Fold 0) ---")
            print(classification_report(va_y, va_preds, target_names=CLASSES, digits=4))

            # Збереження OOF та test логітів (для ансамблю з бейзлайном і майбутніми моделями)
            oof_df = pd.DataFrame({
                "id": va_df["id"].to_numpy(),
                "y_true": va_y,
                "y_pred": va_preds,
                "logit_human": best_oof_logits[:, 0],
                "logit_fusion": best_oof_logits[:, 1],
                "logit_ai": best_oof_logits[:, 2],
            })
            te_preds = te_logits.argmax(axis=1)
            te_out = pd.DataFrame({
                "id": test_df["id"].to_numpy(),
                "y_pred": te_preds,
                "logit_human": te_logits[:, 0],
                "logit_fusion": te_logits[:, 1],
                "logit_ai": te_logits[:, 2],
            })
            oof_df.to_parquet(OUT_DIR / "mdeberta_oof_fold0.parquet", index=False)
            te_out.to_parquet(OUT_DIR / "mdeberta_test_preds.parquet", index=False)

            # Генерація та перевірка сабмішну
            sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
            pred_map = te_out.set_index("id")["y_pred"].map(ID2LABEL)
            sub = pd.DataFrame({"id": sample_sub["id"], "label": sample_sub["id"].map(pred_map)})
            assert len(sub) == 25_000
            assert (sub["id"].to_numpy() == sample_sub["id"].to_numpy()).all()
            assert sub["label"].notna().all()
            assert set(sub["label"].unique()) == set(CLASSES)

            sub_path = OUT_DIR / "submission_mdeberta.csv"
            sub.to_csv(sub_path, index=False)
            print(f"Saved: mdeberta_oof_fold0.parquet, mdeberta_test_preds.parquet, {sub_path.name}")
            print("\n--- Test Predicted Class % ---")
            print((sub["label"].value_counts(normalize=True) * 100).round(2).reindex(CLASSES))

            # Видаляємо важкий .pt (~1.1 GB), якщо не плануєш дотреновувати з нього
            if best_ckpt_path.exists():
                best_ckpt_path.unlink()

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/train_mdeberta.py


In [3]:
!torchrun --nproc_per_node=2 --master_port=29501 /kaggle/working/train_mdeberta.py --smoke_test --epochs 1

W1121 14:58:45.845000 75 torch/distributed/run.py:851] 
W1121 14:58:45.845000 75 torch/distributed/run.py:851] *****************************************
W1121 14:58:45.845000 75 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1121 14:58:45.845000 75 torch/distributed/run.py:851] *****************************************
[SMOKE TEST] GPUs=2 | max_len=512 (head+tail) | per_gpu_bs=8 | grad_accum=2 | eff_bs=32 | lr=2e-05
Rows -> Train: 2000 | Val: 500 | Test: 500
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1121 14:59:10.847989260 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global r

In [4]:
!torchrun --nproc_per_node=2 --master_port=29501 /kaggle/working/train_mdeberta.py --epochs 1

W1121 15:01:33.141000 259 torch/distributed/run.py:851] 
W1121 15:01:33.141000 259 torch/distributed/run.py:851] *****************************************
W1121 15:01:33.141000 259 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1121 15:01:33.141000 259 torch/distributed/run.py:851] *****************************************
[FULL RUN] GPUs=2 | max_len=512 (head+tail) | per_gpu_bs=8 | grad_accum=2 | eff_bs=32 | lr=2e-05
Rows -> Train: 217762 | Val: 54590 | Test: 25000
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1121 15:01:56.954212412 ProcessGroupNCCL.cpp:5188] Guessing device ID based on 